# ImageJ Plugin Java Kernel Check

Java-only notebook to validate the Jupyter Java kernel and run quick diagnostics for the ImageJ plugin project.

## Prerequisites (strict)
- `java` must be on `PATH`
- `javac` must be on `PATH`
- `mvn` (Apache Maven) must be on `PATH`

This notebook now hard-fails preflight if any of those are missing.
It will also:
1. Parse `FFT/pom.xml` and list direct dependencies
2. Check whether dependency jars are present in local Maven repo (`~/.m2/repository`)
3. Resolve missing dependencies (`mvn dependency:resolve`)
4. Run Maven dependency/plugin update audit (`versions:display-*`)
5. Skip the long build check by default unless explicitly enabled in the final cell

In [10]:
System.out.println("Java kernel is running ✅");
System.out.println("java.version: " + System.getProperty("java.version"));
System.out.println("java.vendor:  " + System.getProperty("java.vendor"));
System.out.println("java.home:    " + System.getProperty("java.home"));
System.out.println("user.dir:     " + System.getProperty("user.dir"));

Java kernel is running ✅
java.version: 25.0.2
java.vendor:  Eclipse Adoptium
java.home:    C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot
user.dir:     c:\Users\dunnmk\repos\imgjplugin\notebooks


In [11]:
import java.nio.file.*;

Path findProjectRoot(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 8 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Path root = findProjectRoot(Paths.get(System.getProperty("user.dir")));
if (root == null) {
    throw new RuntimeException("Could not find project root containing FFT/pom.xml from user.dir");
}

System.out.println("Project root: " + root);
System.out.println("FFT pom.xml exists: " + Files.exists(root.resolve("FFT").resolve("pom.xml")));

Path pluginDir = root.resolve("FFT").resolve("src").resolve("fftanalysis").resolve("imagej");
System.out.println("Plugin source dir: " + pluginDir);
System.out.println("Plugin source dir exists: " + Files.isDirectory(pluginDir));

Project root: c:\Users\dunnmk\repos\imgjplugin
FFT pom.xml exists: true
Plugin source dir: c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej
Plugin source dir exists: true


In [12]:
import java.util.*;

List<String> expected = Arrays.asList(
    "FIBA_Tile_Montage.java",
    "FIBA_Orientation.java",
    "FIBA_Orientation_Profile.java",
    "fibaMain.java",
    "FibaMatlabProcessor.java"
);

Path pluginDir2 = root.resolve("FFT").resolve("src").resolve("fftanalysis").resolve("imagej");
for (String f : expected) {
    Path p = pluginDir2.resolve(f);
    System.out.println((Files.exists(p) ? "OK   " : "MISS ") + p);
}

OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\FIBA_Tile_Montage.java
OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\FIBA_Orientation.java
OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\FIBA_Orientation_Profile.java
OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\fibaMain.java
OK   c:\Users\dunnmk\repos\imgjplugin\FFT\src\fftanalysis\imagej\FibaMatlabProcessor.java


In [15]:
import java.io.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;
import java.util.regex.Pattern;
import javax.xml.parsers.*;
import org.w3c.dom.*;

Path findProjectRoot(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 10 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Path findOnPath(String exe) {
    String path = System.getenv("PATH");
    if (path == null || path.isBlank()) return null;

    String sep = File.pathSeparator;
    String[] pathExts;
    if (System.getProperty("os.name").toLowerCase().contains("win")) {
        String pe = System.getenv("PATHEXT");
        pathExts = (pe == null || pe.isBlank()) ? new String[] {".EXE", ".CMD", ".BAT"} : pe.split(";");
    } else {
        pathExts = new String[] {""};
    }

    for (String part : path.split(Pattern.quote(sep))) {
        if (part == null || part.isBlank()) continue;
        Path base = Paths.get(part.trim());
        if (!Files.isDirectory(base)) continue;

        Path direct = base.resolve(exe);
        if (Files.isRegularFile(direct)) return direct;

        for (String ext : pathExts) {
            String e = ext == null ? "" : ext.trim();
            if (!e.isEmpty() && !exe.toLowerCase().endsWith(e.toLowerCase())) {
                Path withExt = base.resolve(exe + e);
                if (Files.isRegularFile(withExt)) return withExt;
            }
        }
    }
    return null;
}

Path findMavenExecutable() {
    Path p = findOnPath("mvn");
    if (p != null) return p;

    String mavenHome = System.getenv("MAVEN_HOME");
    if (mavenHome != null && !mavenHome.isBlank()) {
        Path bin = Paths.get(mavenHome, "bin");
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }

    List<Path> roots = Arrays.asList(
        Paths.get(System.getProperty("user.home"), "tools", "maven", "apache-maven-3.9.6", "bin"),
        Paths.get(System.getProperty("user.home"), "tools", "apache-maven-3.9.6", "bin"),
        Paths.get(System.getProperty("user.home"), ".tools", "apache-maven-3.9.6", "bin")
    );
    for (Path bin : roots) {
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }
    return null;
}

String runAndCollect(List<String> cmd, Path cwd, int timeoutSeconds, int maxLines) throws Exception {
    ProcessBuilder pb = new ProcessBuilder(cmd);
    if (cwd != null) pb.directory(cwd.toFile());
    pb.redirectErrorStream(true);

    Process proc = pb.start();
    StringBuilder sb = new StringBuilder();
    int shown = 0;

    try (BufferedReader br = new BufferedReader(new InputStreamReader(proc.getInputStream(), StandardCharsets.UTF_8))) {
        String line;
        long deadline = System.currentTimeMillis() + timeoutSeconds * 1000L;
        while (true) {
            while (br.ready() && (line = br.readLine()) != null) {
                if (shown < maxLines) {
                    sb.append(line).append("\n");
                    shown++;
                }
            }

            if (!proc.isAlive()) break;
            if (System.currentTimeMillis() > deadline) {
                proc.destroyForcibly();
                throw new RuntimeException("Command timed out after " + timeoutSeconds + "s: " + String.join(" ", cmd));
            }
            Thread.sleep(100);
        }

        while ((line = br.readLine()) != null) {
            if (shown < maxLines) {
                sb.append(line).append("\n");
                shown++;
            }
        }
    }

    int exit = proc.waitFor();
    sb.append("[exit=").append(exit).append("]\n");
    if (exit != 0) {
        throw new RuntimeException("Command failed (exit=" + exit + "): " + String.join(" ", cmd) + "\n" + sb);
    }
    return sb.toString();
}

String childText(Element parent, String tag) {
    NodeList nl = parent.getElementsByTagName(tag);
    if (nl.getLength() == 0) return null;
    return nl.item(0).getTextContent().trim();
}

String resolveProps(String in, Map<String, String> props) {
    if (in == null) return null;
    String out = in;
    int guard = 0;
    while (out.contains("${") && guard++ < 20) {
        int s = out.indexOf("${");
        int e = out.indexOf("}", s + 2);
        if (s < 0 || e < 0) break;
        String key = out.substring(s + 2, e);
        String val = props.getOrDefault(key, "${" + key + "}");
        out = out.substring(0, s) + val + out.substring(e + 1);
    }
    return out;
}

Path root2 = findProjectRoot(Paths.get(System.getProperty("user.dir")));
if (root2 == null) throw new RuntimeException("Could not find project root containing FFT/pom.xml from user.dir");
Path fftDir = root2.resolve("FFT");
Path pom = fftDir.resolve("pom.xml");
if (!Files.exists(pom)) throw new RuntimeException("Missing pom.xml at: " + pom);

System.out.println("Project root: " + root2);
System.out.println("FFT dir:      " + fftDir);
System.out.println("pom.xml:      " + pom);

Path javaPath = findOnPath("java");
Path javacPath = findOnPath("javac");
Path mvnPath = findMavenExecutable();

System.out.println("\nPATH checks:");
System.out.println("java  -> " + (javaPath == null ? "MISSING" : javaPath));
System.out.println("javac -> " + (javacPath == null ? "MISSING" : javacPath));
System.out.println("mvn   -> " + (mvnPath == null ? "MISSING" : mvnPath));

boolean toolsOk = (javaPath != null && javacPath != null && mvnPath != null);
if (!toolsOk) {
    System.out.println("\nPreflight blocked: missing required executable(s). Ensure java/javac/mvn are discoverable.");
} else {
    System.out.println("\nVersion checks:");
    System.out.println(runAndCollect(Arrays.asList(javaPath.toString(), "-version"), fftDir, 30, 30));
    System.out.println(runAndCollect(Arrays.asList(javacPath.toString(), "-version"), fftDir, 30, 30));
    System.out.println(runAndCollect(Arrays.asList(mvnPath.toString(), "-v"), fftDir, 45, 80));

    DocumentBuilderFactory dbf = DocumentBuilderFactory.newInstance();
    dbf.setNamespaceAware(false);
    DocumentBuilder db = dbf.newDocumentBuilder();
    Document doc = db.parse(pom.toFile());

    Map<String, String> props = new LinkedHashMap<>();
    NodeList propNodes = doc.getElementsByTagName("properties");
    if (propNodes.getLength() > 0) {
        Node n = propNodes.item(0);
        NodeList kids = n.getChildNodes();
        for (int i = 0; i < kids.getLength(); i++) {
            Node c = kids.item(i);
            if (c.getNodeType() == Node.ELEMENT_NODE) props.put(c.getNodeName(), c.getTextContent().trim());
        }
    }

    NodeList depNodes = doc.getElementsByTagName("dependency");
    List<String> deps = new ArrayList<>();
    for (int i = 0; i < depNodes.getLength(); i++) {
        Element d = (Element) depNodes.item(i);
        String g = resolveProps(childText(d, "groupId"), props);
        String a = resolveProps(childText(d, "artifactId"), props);
        String v = resolveProps(childText(d, "version"), props);
        String s = resolveProps(childText(d, "scope"), props);
        if (s == null) s = "compile";
        if (g != null && a != null && v != null) deps.add(g + ":" + a + ":" + v + ":" + s);
    }

    System.out.println("Direct dependencies in pom.xml:");
    for (String d : deps) System.out.println("  - " + d);

    Path m2 = Paths.get(System.getProperty("user.home"), ".m2", "repository");
    System.out.println("\nLocal Maven repo: " + m2);
    System.out.println("\nLocal dependency presence (jar):");
    for (String d : deps) {
        String[] p = d.split(":");
        String g = p[0], a = p[1], v = p[2];
        Path jar = m2.resolve(g.replace('.', File.separatorChar)).resolve(a).resolve(v).resolve(a + "-" + v + ".jar");
        System.out.println((Files.isRegularFile(jar) ? "FOUND " : "MISS  ") + d + " -> " + jar);
    }

    System.out.println("\nResolving Maven dependencies now (downloads if missing)...");
    System.out.println(runAndCollect(Arrays.asList(mvnPath.toString(), "-B", "-DskipTests", "dependency:resolve"), fftDir, 600, 250));

    System.out.println("Checking for dependency/plugin updates...");
    System.out.println(runAndCollect(Arrays.asList(mvnPath.toString(), "-B", "versions:display-dependency-updates", "versions:display-plugin-updates"), fftDir, 600, 350));

    System.out.println("\nPreflight complete ✅");
    System.out.println("No build/test command has been run in this cell.");

    boolean RUN_MAVEN_BUILD_CHECK = false;
    if (RUN_MAVEN_BUILD_CHECK) {
        System.out.println("\nRunning bounded Maven build check...");
        System.out.println(runAndCollect(Arrays.asList(mvnPath.toString(), "-B", "-DskipTests", "test"), fftDir, 900, 400));
        System.out.println("Build check finished successfully ✅");
    } else {
        System.out.println("\nBuild check skipped by design (RUN_MAVEN_BUILD_CHECK=false).");
    }
}

Project root: c:\Users\dunnmk\repos\imgjplugin
FFT dir:      c:\Users\dunnmk\repos\imgjplugin\FFT
pom.xml:      c:\Users\dunnmk\repos\imgjplugin\FFT\pom.xml

PATH checks:
java  -> C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot\bin\java.EXE
javac -> C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot\bin\javac.EXE
mvn   -> C:\Users\dunnmk\tools\maven\apache-maven-3.9.6\bin\mvn.cmd

Version checks:
openjdk version "25.0.2" 2026-01-20 LTS
OpenJDK Runtime Environment Temurin-25.0.2+10 (build 25.0.2+10-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.2+10 (build 25.0.2+10-LTS, mixed mode, sharing)
[exit=0]

javac 25.0.2
[exit=0]


Apache Maven 3.9.6 (bc0240f3c744dd6b6ec2920b3cd08dcc295161ae)
Maven home: C:\Users\dunnmk\tools\maven\apache-maven-3.9.6
Java version: 25.0.2, vendor: Eclipse Adoptium, runtime: C:\Users\dunnmk\AppData\Local\Programs\Eclipse Adoptium\jdk-25.0.2.10-hotspot
Default locale: en_US, platform encoding: UTF-8
OS name: "w

## Visualization pipeline (dual-image comparison)

This section runs the Java pipeline for **two input images** and creates two final figure outputs so orientation direction can be compared directly.

### Default quality enhancement (STORM-like)
Before running the pipeline cell, run the preprocessing cell to create **STORM-like cleaned images** (bandpass + spot enhancement).
The pipeline cell now enforces these STORM-like preprocessed images by default.

Inputs:
1. `C:/Users/dunnmk/Downloads/C15D5P001 (1).jpg`
2. `C:/Users/dunnmk/OneDrive - Michigan Medicine/Pictures/Picture1.jpg`

Outputs are written to separate folders under `notebooks/_assets/fiba_tile_montage/` to avoid overlap.

## Mathematical background for preprocessing and orientation pipeline

This workflow estimates local fiber orientation by transforming image intensity into directional descriptors. The biological target is to quantify anisotropy (preferred alignment direction) in fibrillar structures such as collagen-rich extracellular matrix.

### 1) STORM-like enhancement used here
The preprocessing cell applies a Difference-of-Gaussians (DoG), thresholding, and gamma compression:

$$I_{\mathrm{DoG}}(x,y)=\left(G_{\sigma_s}*I\right)(x,y)-\left(G_{\sigma_l}*I\right)(x,y),\quad \sigma_s<\sigma_l$$

$$I_{+}(x,y)=\max\left(0,\,I_{\mathrm{DoG}}(x,y)-\tau\right),\quad \tau=\mu_{\mathrm{DoG}}+k\,\sigma_{\mathrm{DoG}}$$

$$I_{\mathrm{enh}}(x,y)=\left(I_{+}(x,y)\right)^{\gamma}$$

Why this helps biologically: DoG suppresses low-frequency background and emphasizes filament-like features, improving downstream orientation estimates when illumination is uneven or signal is sparse.

### 2) Orientation extraction logic
The Java/ImageJ pipeline then computes directional summaries (FFT/polar/SOL/mask/reconstruction). Conceptually, dominant direction is the angle with strongest directional energy:

$$\theta^*=\arg\max_{\theta}\,E(\theta)$$

where $E(\theta)$ is estimated from transformed domains (e.g., FFT magnitude in polar coordinates). This maps directly to biological hypotheses such as “fibers are more aligned in condition A than B.”

### Representative prior usage in bioimage analysis
- Schindelin et al., **Fiji: an open-source platform for biological-image analysis**, *Nature Methods* (2012).
- Rezakhaniha et al., **Experimental investigation of collagen waviness and orientation in the arterial adventitia using confocal microscopy**, *Biomechanics and Modeling in Mechanobiology* (2012).
- Bredfeldt et al., **Computational segmentation of collagen fibers from second-harmonic generation images of breast cancer**, *Journal of Biomedical Optics* (2014).

In [36]:
import java.awt.image.BufferedImage;
import java.nio.file.*;
import java.util.*;
import javax.imageio.ImageIO;

// -----------------------------------
// STORM-like preprocessing (default)
// -----------------------------------
final double SIGMA_SMALL = 1.0;
final double SIGMA_LARGE = 3.0;
final double THRESH_STD = 0.50;
final double GAMMA = 0.70;

double[][] toGray(BufferedImage img) {
    int h = img.getHeight();
    int w = img.getWidth();
    double[][] g = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int gg = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            g[y][x] = 0.299 * r + 0.587 * gg + 0.114 * b;
        }
    }
    return g;
}

BufferedImage fromGray(double[][] g) {
    int h = g.length;
    int w = g[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(Math.max(0, Math.min(255, g[y][x])));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

double[][] copy2D(double[][] a) {
    int h = a.length, w = a[0].length;
    double[][] b = new double[h][w];
    for (int y = 0; y < h; y++) System.arraycopy(a[y], 0, b[y], 0, w);
    return b;
}

void normalize01(double[][] a) {
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : a) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    for (int y = 0; y < a.length; y++) {
        for (int x = 0; x < a[0].length; x++) {
            a[y][x] = (a[y][x] - min) / span;
        }
    }
}

double[] gaussianKernel1D(double sigma) {
    int radius = Math.max(1, (int)Math.ceil(3.0 * sigma));
    int n = radius * 2 + 1;
    double[] k = new double[n];
    double sum = 0.0;
    for (int i = -radius; i <= radius; i++) {
        double v = Math.exp(-(i * i) / (2.0 * sigma * sigma));
        k[i + radius] = v;
        sum += v;
    }
    for (int i = 0; i < n; i++) k[i] /= sum;
    return k;
}

double[][] convolveHorizontal(double[][] src, double[] k) {
    int h = src.length, w = src[0].length;
    int r = k.length / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int i = -r; i <= r; i++) {
                int xx = Math.min(w - 1, Math.max(0, x + i));
                s += src[y][xx] * k[i + r];
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] convolveVertical(double[][] src, double[] k) {
    int h = src.length, w = src[0].length;
    int r = k.length / 2;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double s = 0.0;
            for (int i = -r; i <= r; i++) {
                int yy = Math.min(h - 1, Math.max(0, y + i));
                s += src[yy][x] * k[i + r];
            }
            out[y][x] = s;
        }
    }
    return out;
}

double[][] gaussianBlur(double[][] src, double sigma) {
    double[] k = gaussianKernel1D(sigma);
    return convolveVertical(convolveHorizontal(src, k), k);
}

double[][] stormLikeEnhance(double[][] gray255) {
    double[][] norm = copy2D(gray255);
    normalize01(norm);

    double[][] gSmall = gaussianBlur(norm, SIGMA_SMALL);
    double[][] gLarge = gaussianBlur(norm, SIGMA_LARGE);

    int h = norm.length, w = norm[0].length;
    double[][] dog = new double[h][w];
    double mean = 0.0;
    double sq = 0.0;
    int n = h * w;

    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            dog[y][x] = gSmall[y][x] - gLarge[y][x];
            mean += dog[y][x];
            sq += dog[y][x] * dog[y][x];
        }
    }
    mean /= n;
    double var = Math.max(0.0, (sq / n) - mean * mean);
    double std = Math.sqrt(var);
    double thresh = mean + THRESH_STD * std;

    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double v = Math.max(0.0, dog[y][x] - thresh);
            v = Math.pow(v, GAMMA);
            out[y][x] = v;
        }
    }

    // Rescale to 0..255
    double min = Double.POSITIVE_INFINITY, max = Double.NEGATIVE_INFINITY;
    for (double[] row : out) for (double v : row) {
        min = Math.min(min, v);
        max = Math.max(max, v);
    }
    double span = Math.max(1e-9, max - min);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            out[y][x] = 255.0 * (out[y][x] - min) / span;
        }
    }
    return out;
}

Path rootPre = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
if (rootPre == null) throw new RuntimeException("Could not locate project root for preprocessing.");
Path preDir = rootPre.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("preprocessed");
Files.createDirectories(preDir);

List<Path> rawInputs = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> rawBases = Arrays.asList("C15D5P001_1", "Picture1");

System.out.println("Running STORM-like preprocessing...");
System.out.println("SIGMA_SMALL=" + SIGMA_SMALL + ", SIGMA_LARGE=" + SIGMA_LARGE + ", THRESH_STD=" + THRESH_STD + ", GAMMA=" + GAMMA);

for (int i = 0; i < rawInputs.size(); i++) {
    Path in = rawInputs.get(i);
    if (!Files.isRegularFile(in)) throw new RuntimeException("Missing input for preprocessing: " + in);

    BufferedImage img = ImageIO.read(in.toFile());
    if (img == null) throw new RuntimeException("Could not read image: " + in);

    double[][] gray = toGray(img);
    double[][] enhanced = stormLikeEnhance(gray);

    Path out = preDir.resolve(rawBases.get(i) + "_clean_storm.jpg");
    ImageIO.write(fromGray(enhanced), "jpg", out.toFile());
    System.out.println("Preprocessed image written: " + out);
}

System.out.println("\nPreprocessing complete ✅ (STORM-like bandpass + spot enhancement)");

Running STORM-like preprocessing...
SIGMA_SMALL=1.0, SIGMA_LARGE=3.0, THRESH_STD=0.5, GAMMA=0.7
Preprocessed image written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\preprocessed\C15D5P001_1_clean_storm.jpg
Preprocessed image written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\preprocessed\Picture1_clean_storm.jpg

Preprocessing complete ✅ (STORM-like bandpass + spot enhancement)


In [40]:
import java.io.*;
import java.nio.charset.StandardCharsets;
import java.nio.file.*;
import java.util.*;
import java.util.regex.Matcher;
import java.util.regex.Pattern;

Path findProjectRootViz(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 12 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Path findOnPathViz(String exe) {
    String path = System.getenv("PATH");
    if (path == null || path.isBlank()) return null;
    String sep = File.pathSeparator;
    String[] exts = System.getProperty("os.name").toLowerCase().contains("win")
        ? new String[] {"", ".cmd", ".bat", ".exe"}
        : new String[] {""};

    for (String part : path.split(Pattern.quote(sep))) {
        if (part == null || part.isBlank()) continue;
        Path dir = Paths.get(part.trim());
        if (!Files.isDirectory(dir)) continue;
        for (String ext : exts) {
            Path cand = dir.resolve(exe + ext);
            if (Files.isRegularFile(cand)) return cand;
        }
    }
    return null;
}

Path findMavenViz() {
    Path p = findOnPathViz("mvn");
    if (p != null) return p;
    String mh = System.getenv("MAVEN_HOME");
    if (mh != null && !mh.isBlank()) {
        Path bin = Paths.get(mh, "bin");
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }
    for (Path bin : Arrays.asList(
            Paths.get(System.getProperty("user.home"), "tools", "maven", "apache-maven-3.9.6", "bin"),
            Paths.get(System.getProperty("user.home"), "tools", "apache-maven-3.9.6", "bin"),
            Paths.get(System.getProperty("user.home"), ".tools", "apache-maven-3.9.6", "bin")
    )) {
        for (String c : new String[] {"mvn.cmd", "mvn.bat", "mvn.exe", "mvn"}) {
            Path m = bin.resolve(c);
            if (Files.isRegularFile(m)) return m;
        }
    }
    return null;
}

String runAndCollectViz(List<String> cmd, Path cwd, int timeoutSeconds, int maxLines) throws Exception {
    ProcessBuilder pb = new ProcessBuilder(cmd);
    pb.directory(cwd.toFile());
    pb.redirectErrorStream(true);
    Process proc = pb.start();

    StringBuilder sb = new StringBuilder();
    int shown = 0;
    long deadline = System.currentTimeMillis() + timeoutSeconds * 1000L;
    try (BufferedReader br = new BufferedReader(new InputStreamReader(proc.getInputStream(), StandardCharsets.UTF_8))) {
        String line;
        while (true) {
            while (br.ready() && (line = br.readLine()) != null) {
                if (shown < maxLines) {
                    sb.append(line).append("\n");
                    shown++;
                }
            }
            if (!proc.isAlive()) break;
            if (System.currentTimeMillis() > deadline) {
                proc.destroyForcibly();
                throw new RuntimeException("Pipeline command timed out: " + String.join(" ", cmd));
            }
            Thread.sleep(100);
        }
        while ((line = br.readLine()) != null) {
            if (shown < maxLines) {
                sb.append(line).append("\n");
                shown++;
            }
        }
    }
    int exit = proc.waitFor();
    sb.append("[exit=").append(exit).append("]\n");
    if (exit != 0) throw new RuntimeException("Command failed: " + String.join(" ", cmd) + "\n" + sb);
    return sb.toString();
}

Path rootViz = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
if (rootViz == null) throw new RuntimeException("Could not locate project root for visualization step.");
Path fftDirViz = rootViz.resolve("FFT");

Path mvnViz = findMavenViz();
if (mvnViz == null) {
    throw new RuntimeException("Maven executable not found, cannot run pipeline generation step.");
}

Path preDir = rootViz.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("preprocessed");
boolean ENFORCE_STORM_PREPROCESSED_DEFAULT = true;

List<Path> inputImages = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> baseNames = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> outputDirs = Arrays.asList(
    rootViz.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootViz.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

for (int idx = 0; idx < inputImages.size(); idx++) {
    Path rawImage = inputImages.get(idx);
    String baseNameViz = baseNames.get(idx);
    Path assetsDirViz = outputDirs.get(idx);

    Path cleanedCandidate = preDir.resolve(baseNameViz + "_clean_storm.jpg");
    Path sourceImageViz = rawImage;
    if (ENFORCE_STORM_PREPROCESSED_DEFAULT) {
        if (!Files.isRegularFile(cleanedCandidate)) {
            throw new RuntimeException("Default mode requires STORM-like preprocessed image, but file is missing: " + cleanedCandidate + "\nRun the preprocessing cell first.");
        }
        sourceImageViz = cleanedCandidate;
    }

    if (!Files.isRegularFile(sourceImageViz)) {
        throw new RuntimeException("Source image not found: " + sourceImageViz + "\nCannot run real pipeline without this file.");
    }
    Files.createDirectories(assetsDirViz);

    System.out.println("\n==================================================");
    System.out.println("Running real tile pipeline from source image...");
    System.out.println("Case:   " + baseNameViz);
    System.out.println("Raw:    " + rawImage);
    System.out.println("Using:  " + sourceImageViz + " (STORM default enforced)");
    System.out.println("Output: " + assetsDirViz);

    List<String> cmd = Arrays.asList(
        mvnViz.toString(),
        "-B",
        "-Dtest=fftanalysis.imagej.GenerateTileMontageFromSourceTest",
        "-Dfiba.input=" + sourceImageViz.toString(),
        "-Dfiba.output=" + assetsDirViz.toString(),
        "-Dfiba.base=" + baseNameViz,
        "-Dfiba.tilesY=10",
        "test"
    );
    System.out.println(runAndCollectViz(cmd, fftDirViz, 900, 220));

    Path boxesPathViz = assetsDirViz.resolve(baseNameViz + "_tile_boxes.jpg");
    Path stackPathViz = assetsDirViz.resolve(baseNameViz + "_tile_montage.jpg");
    Path csvPathViz = assetsDirViz.resolve(baseNameViz + "_tile_results.csv");
    if (!Files.isRegularFile(boxesPathViz) || !Files.isRegularFile(stackPathViz) || !Files.isRegularFile(csvPathViz)) {
        throw new RuntimeException("Pipeline run completed but expected outputs are missing in: " + assetsDirViz);
    }

    Pattern tilePatternViz = Pattern.compile("^" + Pattern.quote(baseNameViz) + "_tile(\\d+)_crop\\.jpg$");
    Set<Integer> idsViz = new LinkedHashSet<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(assetsDirViz, "*_crop.jpg")) {
        for (Path p : ds) {
            Matcher m = tilePatternViz.matcher(p.getFileName().toString());
            if (m.matches()) idsViz.add(Integer.parseInt(m.group(1)));
        }
    }
    List<Integer> tileIdsViz = new ArrayList<>(idsViz);
    tileIdsViz.sort(Comparator.naturalOrder());
    if (tileIdsViz.isEmpty()) throw new RuntimeException("No generated crop tiles found after pipeline run for " + baseNameViz);

    System.out.println("Generated assets confirmed for " + baseNameViz + ":");
    System.out.println("Tiles: " + tileIdsViz);
    System.out.println("Overlay: " + boxesPathViz.getFileName());
    System.out.println("Tile stack: " + stackPathViz.getFileName());
    System.out.println("CSV: " + csvPathViz.getFileName());
}

System.out.println("\nDual-image pipeline generation complete ✅");


Running real tile pipeline from source image...
Case:   C15D5P001_1
Raw:    C:\Users\dunnmk\Downloads\C15D5P001 (1).jpg
Using:  c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\preprocessed\C15D5P001_1_clean_storm.jpg (STORM default enforced)
Output: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source

[INFO] Scanning for projects...
[INFO] 
[INFO] ---------------------< com.mikdunn:imgjplugin-fft >---------------------
[INFO] Building ImageJ FFT Orientation Plugin 0.1.0-SNAPSHOT
[INFO]   from pom.xml
[INFO] --------------------------------[ jar ]---------------------------------
[INFO] 
[INFO] --- enforcer:3.6.1:enforce (enforce-maven) @ imgjplugin-fft ---
[INFO] Rule 0: org.apache.maven.enforcer.rules.version.RequireMavenVersion passed
[INFO] 
[INFO] --- resources:3.3.1:resources (default-resources) @ imgjplugin-fft ---
[INFO] Copying 1 resource from resources to target\classes
[INFO] 
[INFO] --- compiler:3.15.0:compile (def

In [ ]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.geom.Rectangle2D;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;

BufferedImage readOrPlaceholder(Path p, int w, int h, String label) throws IOException {
    if (Files.exists(p)) {
        BufferedImage img = ImageIO.read(p.toFile());
        if (img != null) return img;
    }
    BufferedImage miss = new BufferedImage(w, h, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = miss.createGraphics();
    g.setColor(new Color(245, 245, 245));
    g.fillRect(0, 0, w, h);
    g.setColor(new Color(200, 60, 60));
    g.setFont(new Font("SansSerif", Font.BOLD, 18));
    g.drawString("MISSING", 16, 30);
    g.setColor(Color.DARK_GRAY);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));
    g.drawString(label, 16, 54);
    g.dispose();
    return miss;
}

void drawFit(Graphics2D g, BufferedImage src, int x, int y, int w, int h) {
    double sx = w / (double) src.getWidth();
    double sy = h / (double) src.getHeight();
    double s = Math.min(sx, sy);
    int nw = Math.max(1, (int) Math.round(src.getWidth() * s));
    int nh = Math.max(1, (int) Math.round(src.getHeight() * s));
    int ox = x + (w - nw) / 2;
    int oy = y + (h - nh) / 2;
    g.drawImage(src, ox, oy, nw, nh, null);
}

int drawSectionTitle(Graphics2D g, String title, int x, int y, int w) {
    g.setColor(new Color(24, 44, 74));
    g.setFont(new Font("SansSerif", Font.BOLD, 22));
    g.drawString(title, x, y + 22);
    g.setColor(new Color(220, 226, 235));
    g.fill(new Rectangle2D.Double(x, y + 28, w, 2));
    return y + 36;
}

int drawSingleImageSection(Graphics2D g, String title, Path p, int x, int y, int w, int h) throws IOException {
    int yy = drawSectionTitle(g, title, x, y, w);
    g.setColor(new Color(232, 232, 232));
    g.fillRect(x - 1, yy - 1, w + 2, h + 2);
    BufferedImage img = readOrPlaceholder(p, w, h, p.getFileName().toString());
    drawFit(g, img, x, yy, w, h);
    return yy + h + 18;
}

int drawGridSection(Graphics2D g, String title, List<Integer> ids, String key, Path assetsDir, String baseName, int x, int y, int columns, int cellW, int cellH, int gap) throws IOException {
    int yy = drawSectionTitle(g, title, x, y, columns * cellW + (columns - 1) * gap);
    int rows = (int) Math.ceil(ids.size() / (double) columns);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));

    for (int i = 0; i < ids.size(); i++) {
        int id = ids.get(i);
        int r = i / columns;
        int c = i % columns;
        int cx = x + c * (cellW + gap);
        int cy = yy + r * (cellH + 26 + gap);

        Path p = assetsDir.resolve(baseName + "_tile" + id + "_" + key + ".jpg");
        BufferedImage img = readOrPlaceholder(p, cellW, cellH, p.getFileName().toString());

        g.setColor(new Color(30, 30, 30));
        g.drawString("Tile " + id, cx + 4, cy + 14);
        g.setColor(new Color(232, 232, 232));
        g.fillRect(cx - 1, cy + 17 - 1, cellW + 2, cellH + 2);
        drawFit(g, img, cx, cy + 17, cellW, cellH);
    }

    return yy + rows * (cellH + 26 + gap) + 8;
}

Path rootViz2 = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
if (rootViz2 == null) throw new RuntimeException("Could not locate project root for final figure composition.");

List<Path> inputImages2 = Arrays.asList(
    Paths.get("C:\\Users\\dunnmk\\Downloads\\C15D5P001 (1).jpg"),
    Paths.get("C:\\Users\\dunnmk\\OneDrive - Michigan Medicine\\Pictures\\Picture1.jpg")
);
List<String> baseNames2 = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> outputDirs2 = Arrays.asList(
    rootViz2.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootViz2.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

for (int caseIdx = 0; caseIdx < baseNames2.size(); caseIdx++) {
    String baseNameViz2 = baseNames2.get(caseIdx);
    Path assetsDirViz2 = outputDirs2.get(caseIdx);
    Path originalPath = inputImages2.get(caseIdx);

    Path tileStackPath = assetsDirViz2.resolve(baseNameViz2 + "_tile_montage.jpg");
    Path tileBoxesPath = assetsDirViz2.resolve(baseNameViz2 + "_tile_boxes.jpg");
    if (!Files.isRegularFile(originalPath)) throw new RuntimeException("Missing source image: " + originalPath);
    if (!Files.isRegularFile(tileStackPath)) throw new RuntimeException("Missing generated tile stack: " + tileStackPath + "\nRun previous pipeline cell first.");
    if (!Files.isRegularFile(tileBoxesPath)) throw new RuntimeException("Missing generated tile boxes output: " + tileBoxesPath + "\nRun previous pipeline cell first.");

    Pattern tilePatternViz2 = Pattern.compile("^" + Pattern.quote(baseNameViz2) + "_tile(\\d+)_crop\\.jpg$");
    List<Integer> tileIdsViz2 = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(assetsDirViz2, "*_crop.jpg")) {
        for (Path p : ds) {
            Matcher m = tilePatternViz2.matcher(p.getFileName().toString());
            if (m.matches()) tileIdsViz2.add(Integer.parseInt(m.group(1)));
        }
    }
    tileIdsViz2.sort(Comparator.naturalOrder());
    if (tileIdsViz2.isEmpty()) throw new RuntimeException("No generated tile images found in " + assetsDirViz2);

    List<Integer> firstTen = tileIdsViz2.stream().filter(i -> i >= 1 && i <= 10).toList();
    if (firstTen.isEmpty()) firstTen = tileIdsViz2;

    int margin = 28;
    int pageW = 1320;
    int usableW = pageW - 2 * margin;
    int columns = 5;
    int gap = 14;
    int cellW = (usableW - (columns - 1) * gap) / columns;
    int cellH = 165;
    int rows = (int) Math.ceil(firstTen.size() / (double) columns);
    int gridH = rows * (cellH + 26 + gap) + 8;

    int hOriginal = 380;
    int hTileStack = 330;
    int titleTop = 62;
    int bottomPad = 28;
    int sectionGap = 8;

    int pageH = titleTop
        + (36 + hOriginal + 18)
        + sectionGap
        + (36 + hTileStack + 18)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + sectionGap
        + (36 + gridH)
        + bottomPad;

    BufferedImage canvas = new BufferedImage(pageW, pageH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = canvas.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setRenderingHint(RenderingHints.KEY_INTERPOLATION, RenderingHints.VALUE_INTERPOLATION_BILINEAR);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, pageW, pageH);

    g.setColor(new Color(12, 33, 64));
    g.setFont(new Font("SansSerif", Font.BOLD, 30));
    g.drawString("Final FIBA Figure (Generated from Source): " + baseNameViz2, margin, 38);
    g.setFont(new Font("SansSerif", Font.PLAIN, 14));
    g.drawString("Order: source image -> tile stack -> images 1-10 -> FFT -> polar -> SOL -> mask -> reconstruction", margin, 56);

    int y = titleTop;
    y = drawSingleImageSection(g, "1) Source image", originalPath, margin, y, usableW, hOriginal);
    y += sectionGap;
    y = drawSingleImageSection(g, "2) Tiles stack (generated by pipeline)", tileStackPath, margin, y, usableW, hTileStack);
    y += sectionGap;
    y = drawGridSection(g, "3) Images numbered 1-10", firstTen, "crop", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "4) FFT image", firstTen, "fft", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "5) Polar coordinates", firstTen, "polar", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "6) SOL graph", firstTen, "sol", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "7) Mask", firstTen, "mask", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);
    y += sectionGap;
    y = drawGridSection(g, "8) Reconstruction", firstTen, "rec", assetsDirViz2, baseNameViz2, margin, y, columns, cellW, cellH, gap);

    g.dispose();

    Path outPath = assetsDirViz2.resolve(baseNameViz2 + "_final_figure_java.png");
    ImageIO.write(canvas, "png", outPath.toFile());
    System.out.println("Final figure written: " + outPath);
    System.out.println("Source image used: " + originalPath);
    System.out.println("Tile boxes were generated by pipeline: " + tileBoxesPath + " (exists=" + Files.exists(tileBoxesPath) + ")");
}

System.out.println("\nDual final-figure generation complete ✅");

Final figure written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_final_figure_java.png
Source image used: C:\Users\dunnmk\Downloads\C15D5P001 (1).jpg
Tile boxes were generated by pipeline: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_tile_boxes.jpg (exists=true)
Final figure written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source_picture1\Picture1_final_figure_java.png
Source image used: C:\Users\dunnmk\OneDrive - Michigan Medicine\Pictures\Picture1.jpg
Tile boxes were generated by pipeline: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source_picture1\Picture1_tile_boxes.jpg (exists=true)

Dual final-figure generation complete ✅


In [ ]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

double maskCoverage(BufferedImage img) {
    int w = img.getWidth();
    int h = img.getHeight();
    long on = 0L;
    long total = (long) w * h;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int g = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            int gray = (r + g + b) / 3;
            if (gray > 16) on++;
        }
    }
    return total == 0 ? 0.0 : (100.0 * on / (double) total);
}

BufferedImage readMaskStrict(Path p) throws Exception {
    if (!Files.isRegularFile(p)) {
        throw new RuntimeException("Missing mask image (expected graph input): " + p);
    }
    BufferedImage img = ImageIO.read(p.toFile());
    if (img == null) throw new RuntimeException("Could not decode mask image: " + p);
    return img;
}

Path rootMask = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
if (rootMask == null) throw new RuntimeException("Could not locate project root for mask visualization.");

List<String> maskBases = Arrays.asList("C15D5P001_1", "Picture1");
List<Path> maskDirs = Arrays.asList(
    rootMask.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootMask.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

for (int caseIdx = 0; caseIdx < maskBases.size(); caseIdx++) {
    String base = maskBases.get(caseIdx);
    Path dir = maskDirs.get(caseIdx);

    Pattern tilePattern = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_mask\\.jpg$");
    List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(dir, "*_mask.jpg")) {
        for (Path p : ds) {
            Matcher m = tilePattern.matcher(p.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    if (ids.isEmpty()) throw new RuntimeException("No mask files found for " + base + " in " + dir);

    List<Integer> firstTen = ids.stream().filter(i -> i >= 1 && i <= 10).toList();
    if (firstTen.isEmpty()) firstTen = ids;

    int cols = 5, gap = 14, margin = 22;
    int cellW = 210, cellH = 165;
    int rows = (int)Math.ceil(firstTen.size() / (double)cols);
    int pageW = margin * 2 + cols * cellW + (cols - 1) * gap;
    int pageH = 90 + rows * (cellH + 44 + gap) + 28;

    BufferedImage canvas = new BufferedImage(pageW, pageH, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = canvas.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, pageW, pageH);

    g.setColor(new Color(20, 36, 68));
    g.setFont(new Font("SansSerif", Font.BOLD, 24));
    g.drawString("Mask panel check: " + base, margin, 34);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));
    g.drawString("Each tile is loaded from *_mask.jpg. Coverage shown under tile.", margin, 56);

    double sumCov = 0.0;
    for (int i = 0; i < firstTen.size(); i++) {
        int tile = firstTen.get(i);
        int r = i / cols;
        int c = i % cols;
        int x = margin + c * (cellW + gap);
        int y = 74 + r * (cellH + 44 + gap);

        Path maskPath = dir.resolve(base + "_tile" + tile + "_mask.jpg");
        BufferedImage mask = readMaskStrict(maskPath);
        double cov = maskCoverage(mask);
        sumCov += cov;

        g.setColor(new Color(35, 35, 35));
        g.setFont(new Font("SansSerif", Font.BOLD, 13));
        g.drawString("Tile " + tile, x + 2, y + 14);

        g.setColor(new Color(232, 232, 232));
        g.fillRect(x - 1, y + 18 - 1, cellW + 2, cellH + 2);
        drawFit(g, mask, x, y + 18, cellW, cellH);

        g.setColor(new Color(120, 18, 18));
        g.setFont(new Font("SansSerif", Font.PLAIN, 12));
        g.drawString(String.format(Locale.US, "mask coverage: %.2f%%", cov), x + 2, y + 18 + cellH + 16);
    }

    g.dispose();

    Path out = dir.resolve(base + "_mask_panel_check.png");
    ImageIO.write(canvas, "png", out.toFile());

    double meanCov = sumCov / firstTen.size();
    System.out.printf(Locale.US, "%n%s mask panel written: %s%n", base, out);
    System.out.printf(Locale.US, "%s mean mask coverage over shown tiles: %.2f%%%n", base, meanCov);
}

System.out.println("\nMask graphing verification complete for BOTH images ✅");


C15D5P001_1 mask panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_mask_panel_check.png
C15D5P001_1 mean mask coverage over shown tiles: 10.24%

Picture1 mask panel written: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source_picture1\Picture1_mask_panel_check.png
Picture1 mean mask coverage over shown tiles: 16.01%

Mask graphing verification complete for BOTH images ✅


## How to interpret mask coverage and angle comparison biologically

The mask panel computes the fraction of pixels above threshold in each tile (reported as mask coverage). This can be interpreted as a proxy for how much of the tile participates in coherent directional structure.

$$\mathrm{Coverage}(\%) = 100\times \frac{\#\{(x,y): M(x,y)>t\}}{N_{\mathrm{pixels}}}$$

In the comparison cell, angle differences are wrapped modulo $180^\circ$ because fiber axes are undirected (a line at $\theta$ is equivalent to $\theta+180^\circ$):

$$\Delta\theta_{180}=\left|\mathrm{wrap}_{[-90^\circ,90^\circ]}\left(\theta_B-\theta_A\right)\right|$$

Biological application: lower mean $\Delta\theta_{180}$ implies similar dominant alignment across specimens/conditions, while larger values suggest remodeling or distinct structural organization.

### Representative literature context
- Orientation-based collagen/fiber quantification in vascular and stromal tissues is commonly interpreted as a marker of remodeling and mechanics (e.g., Rezakhaniha et al., 2012).
- Directional texture/statistical orientation metrics are widely used in biomedical microscopy and histology pipelines for phenotype stratification (e.g., Bredfeldt et al., 2014).

## Mask stage verification (both images)

Run the previous Java cell to generate explicit mask-panel outputs for both datasets:

- `_assets/fiba_tile_montage/generated_from_source/C15D5P001_1_mask_panel_check.png`
- `_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_mask_panel_check.png`

C15 mask panel:

![](_assets/fiba_tile_montage/generated_from_source/C15D5P001_1_mask_panel_check.png)

Picture1 mask panel:

![](_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_mask_panel_check.png)

In [3]:
import java.nio.file.*;
import java.nio.charset.StandardCharsets;
import java.util.*;

Path findProjectRootCmp(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 12 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Map<Integer, Double> loadAnglesForComparison(Path csv) throws Exception {
    Map<Integer, Double> out = new LinkedHashMap<>();
    java.util.List<String> lines = Files.readAllLines(csv, StandardCharsets.UTF_8);
    if (lines.isEmpty()) return out;

    String[] header = lines.get(0).split(",");
    int idCol = -1;
    int fiberCol = -1;
    int adjCol = -1;
    for (int i = 0; i < header.length; i++) {
        String h = header[i].trim();
        if ("tile_id".equals(h)) idCol = i;
        if ("pAng_fiber_axis".equals(h)) fiberCol = i;
        if ("pAng_adj".equals(h)) adjCol = i;
    }
    if (idCol < 0) {
        throw new RuntimeException("CSV missing required column tile_id: " + csv);
    }
    int useCol = (fiberCol >= 0) ? fiberCol : adjCol;
    if (useCol < 0) {
        throw new RuntimeException("CSV missing angle columns (need pAng_fiber_axis or pAng_adj): " + csv);
    }

    for (int i = 1; i < lines.size(); i++) {
        String line = lines.get(i).trim();
        if (line.isEmpty()) continue;
        String[] cols = line.split(",");
        if (cols.length <= Math.max(idCol, useCol)) continue;
        int id = Integer.parseInt(cols[idCol].trim());
        double a = Double.parseDouble(cols[useCol].trim());
        out.put(id, a);
    }
    return out;
}

double wrap180(double a) {
    double x = a % 180.0;
    if (x < -90.0) x += 180.0;
    if (x > 90.0) x -= 180.0;
    return x;
}

Path rootCmp = findProjectRootCmp(Paths.get(System.getProperty("user.dir")));
if (rootCmp == null) {
    throw new RuntimeException("Could not find project root containing FFT/pom.xml");
}
Path csvA = rootCmp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source").resolve("C15D5P001_1_tile_results.csv");
Path csvB = rootCmp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1").resolve("Picture1_tile_results.csv");

if (!Files.isRegularFile(csvA) || !Files.isRegularFile(csvB)) {
    throw new RuntimeException("Missing comparison CSV(s). Run the previous pipeline cell first.\nA=" + csvA + "\nB=" + csvB);
}

Map<Integer, Double> a = loadAnglesForComparison(csvA);
Map<Integer, Double> b = loadAnglesForComparison(csvB);

Set<Integer> common = new TreeSet<>(a.keySet());
common.retainAll(b.keySet());
if (common.isEmpty()) throw new RuntimeException("No common tile ids between comparison CSV files.");

System.out.println("Orientation comparison (deg)");
System.out.println("A = C15D5P001_1, B = Picture1");
System.out.println("Using pAng_fiber_axis when present, otherwise pAng_adj.");
System.out.println("tile\tA\tB\t(B-A)\taxis_diff_mod180");

double sumDelta = 0.0;
double sumAxisDiff = 0.0;
for (int t : common) {
    double av = a.get(t);
    double bv = b.get(t);
    double delta = bv - av;
    double axisDiff = Math.abs(wrap180(delta));
    sumDelta += delta;
    sumAxisDiff += axisDiff;
    System.out.printf(Locale.US, "%d\t%.1f\t%.1f\t%.1f\t%.1f%n", t, av, bv, delta, axisDiff);
}

double n = common.size();
System.out.printf(Locale.US, "\nMean delta (B-A): %.2f deg%n", sumDelta / n);
System.out.printf(Locale.US, "Mean axis diff mod 180: %.2f deg%n", sumAxisDiff / n);

Orientation comparison (deg)
A = C15D5P001_1, B = Picture1
Using pAng_fiber_axis when present, otherwise pAng_adj.
tile	A	B	(B-A)	axis_diff_mod180
1	65.0	40.0	-25.0	25.0
2	65.0	37.0	-28.0	28.0
3	68.0	29.0	-39.0	39.0
4	67.0	26.0	-41.0	41.0
5	83.0	29.0	-54.0	54.0
6	88.0	29.0	-59.0	59.0
7	92.0	31.0	-61.0	61.0
8	102.0	28.0	-74.0	74.0
9	107.0	26.0	-81.0	81.0
10	109.0	26.0	-83.0	83.0

Mean delta (B-A): -54.50 deg
Mean axis diff mod 180: 54.50 deg


## Circular statistics (axial orientation)

This next cell summarizes per-tile orientation angles using **axial** circular statistics (appropriate for fiber axes where $\theta \equiv \theta+180^\circ$).

- **Mean direction**: central orientation (mod $180^\circ$) using doubled-angle averaging.
- **Mean resultant length** $R\in[0,1]$: concentration; higher $R$ means tighter alignment.
- **Circular variance** $V=1-R$: dispersion; higher $V$ means broader orientation spread.
- **Circular standard deviation**: spread in degrees on the axial domain.

For methods reporting, include $n$, mean direction, $R$, and $V$ for each condition, plus the same stats for cross-condition tile-wise angle differences when relevant.

In [9]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.nio.charset.StandardCharsets;
import java.util.ArrayList;
import java.util.Collection;
import java.util.LinkedHashMap;
import java.util.Locale;
import java.util.Map;
import java.util.Set;
import java.util.TreeSet;

Path findProjectRootCirc(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 12 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

Map<Integer, Double> loadAnglesForCircular(Path csv) throws Exception {
    Map<Integer, Double> out = new LinkedHashMap<>();
    java.util.List<String> lines = Files.readAllLines(csv, StandardCharsets.UTF_8);
    if (lines.isEmpty()) return out;

    String[] header = lines.get(0).split(",");
    int idCol = -1;
    int fiberCol = -1;
    int adjCol = -1;
    for (int i = 0; i < header.length; i++) {
        String h = header[i].trim();
        if ("tile_id".equals(h)) idCol = i;
        if ("pAng_fiber_axis".equals(h)) fiberCol = i;
        if ("pAng_adj".equals(h)) adjCol = i;
    }
    if (idCol < 0) throw new RuntimeException("CSV missing tile_id: " + csv);
    int useCol = (fiberCol >= 0) ? fiberCol : adjCol;
    if (useCol < 0) throw new RuntimeException("CSV missing pAng_fiber_axis/pAng_adj: " + csv);

    for (int i = 1; i < lines.size(); i++) {
        String line = lines.get(i).trim();
        if (line.isEmpty()) continue;
        String[] cols = line.split(",");
        if (cols.length <= Math.max(idCol, useCol)) continue;
        int id = Integer.parseInt(cols[idCol].trim());
        double a = Double.parseDouble(cols[useCol].trim());
        out.put(id, a);
    }
    return out;
}

double wrap180Circ(double a) {
    double x = a % 180.0;
    if (x < -90.0) x += 180.0;
    if (x > 90.0) x -= 180.0;
    return x;
}

record AxialCircularStats(int n, double meanDeg, double R, double circVariance, double circStdDeg) {}

AxialCircularStats axialStats(Collection<Double> anglesDeg) {
    int n = anglesDeg.size();
    if (n == 0) return new AxialCircularStats(0, Double.NaN, Double.NaN, Double.NaN, Double.NaN);

    double C = 0.0, S = 0.0;
    for (double deg : anglesDeg) {
        double rad2 = Math.toRadians(2.0 * deg);
        C += Math.cos(rad2);
        S += Math.sin(rad2);
    }
    C /= n;
    S /= n;

    double R = Math.sqrt(C * C + S * S);
    double meanRad = 0.5 * Math.atan2(S, C);
    double meanDeg = Math.toDegrees(meanRad);
    if (meanDeg < 0) meanDeg += 180.0;

    double circVariance = 1.0 - R;
    double circStdRad = (R <= 1e-12) ? Double.POSITIVE_INFINITY : 0.5 * Math.sqrt(Math.max(0.0, -2.0 * Math.log(R)));
    double circStdDeg = Double.isFinite(circStdRad) ? Math.toDegrees(circStdRad) : Double.POSITIVE_INFINITY;

    return new AxialCircularStats(n, meanDeg, R, circVariance, circStdDeg);
}

int[] histogram180(Collection<Double> anglesDeg, int bins) {
    int[] h = new int[bins];
    for (double a : anglesDeg) {
        double x = a % 180.0;
        if (x < 0) x += 180.0;
        int bi = (int)Math.floor((x / 180.0) * bins);
        if (bi >= bins) bi = bins - 1;
        h[bi]++;
    }
    return h;
}

void writeAxialHistogram(Path outFile, String title, int[] counts, AxialCircularStats s) throws Exception {
    int W = 980, H = 470;
    int left = 70, right = 24, top = 78, bottom = 80;
    int plotW = W - left - right;
    int plotH = H - top - bottom;
    int bins = counts.length;

    BufferedImage img = new BufferedImage(W, H, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = img.createGraphics();
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, W, H);
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);

    g.setColor(new Color(12, 33, 64));
    g.setFont(new Font("SansSerif", Font.BOLD, 22));
    g.drawString(title, 18, 34);
    g.setFont(new Font("SansSerif", Font.PLAIN, 13));
    g.drawString(String.format(Locale.US, "n=%d   mean=%.2f°   R=%.4f   var=%.4f   circStd=%.2f°", s.n(), s.meanDeg(), s.R(), s.circVariance(), s.circStdDeg()), 18, 56);

    int max = 1;
    for (int c : counts) max = Math.max(max, c);

    g.setColor(new Color(90, 90, 90));
    g.drawLine(left, top + plotH, left + plotW, top + plotH);
    g.drawLine(left, top, left, top + plotH);

    int gap = 8;
    int barW = (plotW - (bins - 1) * gap) / bins;
    for (int i = 0; i < bins; i++) {
        int h = (int)Math.round((counts[i] / (double)max) * (plotH - 8));
        int x = left + i * (barW + gap);
        int y = top + plotH - h;

        g.setColor(new Color(78, 139, 232));
        g.fillRect(x, y, barW, h);

        g.setColor(Color.DARK_GRAY);
        g.setFont(new Font("SansSerif", Font.PLAIN, 11));
        double startDeg = (180.0 * i) / bins;
        g.drawString(String.format(Locale.US, "%.0f°", startDeg), x + Math.max(2, barW / 2 - 12), top + plotH + 18);
        g.drawString(Integer.toString(counts[i]), x + Math.max(2, barW / 2 - 5), y - 4);
    }

    int meanX = left + (int)Math.round((s.meanDeg() / 180.0) * plotW);
    g.setColor(new Color(200, 30, 30));
    g.drawLine(meanX, top, meanX, top + plotH);
    g.setFont(new Font("SansSerif", Font.BOLD, 12));
    g.drawString(String.format(Locale.US, "mean %.1f°", s.meanDeg()), Math.min(meanX + 6, W - 120), top + 14);

    g.setColor(new Color(70, 70, 70));
    g.setFont(new Font("SansSerif", Font.PLAIN, 12));
    g.drawString("Axial orientation bins (0° to <180°)", left + plotW / 2 - 90, H - 22);

    g.dispose();
    ImageIO.write(img, "png", outFile.toFile());
}

Path rootCirc = findProjectRootCirc(Paths.get(System.getProperty("user.dir")));
if (rootCirc == null) throw new RuntimeException("Could not find project root containing FFT/pom.xml");

Path csvA = rootCirc.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source").resolve("C15D5P001_1_tile_results.csv");
Path csvB = rootCirc.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1").resolve("Picture1_tile_results.csv");
if (!Files.isRegularFile(csvA) || !Files.isRegularFile(csvB)) {
    throw new RuntimeException("Missing tile_results.csv files. Run the pipeline generation cells first.\nA=" + csvA + "\nB=" + csvB);
}

Map<Integer, Double> a = loadAnglesForCircular(csvA);
Map<Integer, Double> b = loadAnglesForCircular(csvB);

AxialCircularStats sa = axialStats(a.values());
AxialCircularStats sb = axialStats(b.values());

Set<Integer> common = new TreeSet<>(a.keySet());
common.retainAll(b.keySet());
java.util.List<Double> deltasAbs = new ArrayList<>();
for (int t : common) deltasAbs.add(Math.abs(wrap180Circ(b.get(t) - a.get(t))));
AxialCircularStats sd = axialStats(deltasAbs);

System.out.println("Axial circular statistics (angles modulo 180°)");
System.out.println("Definitions:");
System.out.println("  mean direction uses doubled-angle method");
System.out.println("  R = mean resultant length (0..1), larger means stronger alignment");
System.out.println("  circular variance = 1 - R");
System.out.println();
System.out.println("dataset\tn\tmean_deg\tR\tcirc_variance\tcirc_std_deg");
System.out.printf(Locale.US, "C15D5P001_1\t%d\t%.3f\t%.4f\t%.4f\t%.3f%n", sa.n(), sa.meanDeg(), sa.R(), sa.circVariance(), sa.circStdDeg());
System.out.printf(Locale.US, "Picture1\t%d\t%.3f\t%.4f\t%.4f\t%.3f%n", sb.n(), sb.meanDeg(), sb.R(), sb.circVariance(), sb.circStdDeg());
System.out.printf(Locale.US, "|Δ(B-A)| common tiles\t%d\t%.3f\t%.4f\t%.4f\t%.3f%n", sd.n(), sd.meanDeg(), sd.R(), sd.circVariance(), sd.circStdDeg());

Path outDir = rootCirc.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("circular_stats");
Files.createDirectories(outDir);
int bins = 12;

Path plotA = outDir.resolve("C15D5P001_1_axial_hist.png");
Path plotB = outDir.resolve("Picture1_axial_hist.png");
Path plotD = outDir.resolve("DeltaAbs_B_minus_A_axial_hist.png");

writeAxialHistogram(plotA, "Axial Orientation Histogram: C15D5P001_1", histogram180(a.values(), bins), sa);
writeAxialHistogram(plotB, "Axial Orientation Histogram: Picture1", histogram180(b.values(), bins), sb);
writeAxialHistogram(plotD, "Axial Histogram: |Δ(B-A)| on common tiles", histogram180(deltasAbs, bins), sd);

System.out.println();
System.out.println("Circular-statistics plots written:");
System.out.println("- " + plotA);
System.out.println("- " + plotB);
System.out.println("- " + plotD);
System.out.println();
System.out.println("Interpretation tips:");
System.out.println("- Higher R => tighter orientation concentration.");
System.out.println("- Higher circular variance => more dispersed orientation.");
System.out.println("- For |Δ(B-A)|, lower mean/variance indicates better cross-image orientation agreement.");

Axial circular statistics (angles modulo 180°)
Definitions:
  mean direction uses doubled-angle method
  R = mean resultant length (0..1), larger means stronger alignment
  circular variance = 1 - R

dataset	n	mean_deg	R	circ_variance	circ_std_deg
C15D5P001_1	10	84.455	0.8352	0.1648	17.195
Picture1	10	30.078	0.9876	0.0124	4.526
|Δ(B-A)| common tiles	10	54.521	0.7751	0.2249	20.452

Circular-statistics plots written:
- c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\circular_stats\C15D5P001_1_axial_hist.png
- c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\circular_stats\Picture1_axial_hist.png
- c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\circular_stats\DeltaAbs_B_minus_A_axial_hist.png

Interpretation tips:
- Higher R => tighter orientation concentration.
- Higher circular variance => more dispersed orientation.
- For |Δ(B-A)|, lower mean/variance indicates better cross-image orientation agreement.


## Circular-statistics plots

These plots are generated by the circular-statistics code cell above and saved under `_assets/fiba_tile_montage/circular_stats/`.

### C15D5P001_1 axial orientation histogram

![](_assets/fiba_tile_montage/circular_stats/C15D5P001_1_axial_hist.png)

### Picture1 axial orientation histogram

![](_assets/fiba_tile_montage/circular_stats/Picture1_axial_hist.png)

### Absolute cross-image difference histogram ($|\Delta(B-A)|$ on common tiles)

![](_assets/fiba_tile_montage/circular_stats/DeltaAbs_B_minus_A_axial_hist.png)

In [47]:
Path rootDisp = findProjectRootViz(Paths.get(System.getProperty("user.dir")));
Path figA = rootDisp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source").resolve("C15D5P001_1_final_figure_java.png");
Path figB = rootDisp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1").resolve("Picture1_final_figure_java.png");

System.out.println("Figure A exists: " + Files.exists(figA) + " -> " + figA);
System.out.println("Figure B exists: " + Files.exists(figB) + " -> " + figB);
if (Files.exists(figB)) {
    System.out.println("Figure B size(bytes): " + Files.size(figB));
}

Figure A exists: true -> c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_final_figure_java.png
Figure B exists: true -> c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source_picture1\Picture1_final_figure_java.png
Figure B size(bytes): 1398838


## Final figures (side-by-side comparison set)

Figure A (C15D5P001):

<img src="file:///C:/Users/dunnmk/repos/imgjplugin/notebooks/_assets/fiba_tile_montage/generated_from_source/C15D5P001_1_final_figure_java.png" alt="Figure A" width="100%" />

Fallback relative path:

![](_assets/fiba_tile_montage/generated_from_source/C15D5P001_1_final_figure_java.png)

Figure B (Picture1):

<img src="file:///C:/Users/dunnmk/repos/imgjplugin/notebooks/_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_final_figure_java.png" alt="Figure B" width="100%" />

Fallback relative path:

![](_assets/fiba_tile_montage/generated_from_source_picture1/Picture1_final_figure_java.png)

## Notes on practical use in this biological context

- Use the same preprocessing parameters ($\sigma_s,\sigma_l,k,\gamma$) across groups when comparing orientation outcomes; changing them can alter apparent anisotropy.
- Prefer reporting both central tendency (mean/median orientation) and dispersion (e.g., circular spread) per image or per tile set.
- Keep voxel/pixel size and acquisition settings in metadata; orientation estimates can be biased by resolution and point-spread differences.



## FFT mask family comparison (extended windows + quick scoring)

This section now compares:

- **Poisson**
- **2D Hann**
- **2D Hamming**
- **2D Blackman**
- **2D Blackman–Harris**
- **2D Nuttall**
- **2D Kaiser** (tunable $\beta$)
- **2D Tukey** (tunable $\alpha$)
- **Both** = Poisson $\times$ Tukey
- **2D Gaussian** (radial)

Outputs include:
- Per-dataset mask panels
- Per-dataset masked-tile panels
- **Quick score table** and ranking per dataset
- **Cross-dataset mean quick score** ranking for fast comparison

Artifacts are written to:
- `_assets/fiba_tile_montage/filter_mask_comparison/`

In [13]:
import java.awt.Color;
import java.awt.Font;
import java.awt.Graphics2D;
import java.awt.RenderingHints;
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

Path findProjectRootMaskCmp(Path start) {
    Path p = start.toAbsolutePath().normalize();
    for (int i = 0; i < 12 && p != null; i++) {
        if (Files.exists(p.resolve("FFT").resolve("pom.xml"))) return p;
        p = p.getParent();
    }
    return null;
}

double[][] toGray01(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] g = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int gg = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            g[y][x] = (0.299 * r + 0.587 * gg + 0.114 * b) / 255.0;
        }
    }
    return g;
}

BufferedImage gray01ToImage(double[][] g) {
    int h = g.length, w = g[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_BYTE_GRAY);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int v = (int)Math.round(255.0 * Math.max(0.0, Math.min(1.0, g[y][x])));
            int rgb = (v << 16) | (v << 8) | v;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

BufferedImage colorizeMask(double[][] m) {
    int h = m.length, w = m[0].length;
    BufferedImage out = new BufferedImage(w, h, BufferedImage.TYPE_INT_RGB);
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double v = Math.max(0.0, Math.min(1.0, m[y][x]));
            int r = (int)Math.round(20 + 235 * v);
            int g = (int)Math.round(30 + 200 * v);
            int b = (int)Math.round(70 + 140 * v);
            int rgb = (r << 16) | (g << 8) | b;
            out.setRGB(x, y, rgb);
        }
    }
    return out;
}

double i0Bessel(double x) {
    double ax = Math.abs(x);
    if (ax < 3.75) {
        double y = (x / 3.75);
        y = y * y;
        return 1.0 + y * (3.5156229 + y * (3.0899424 + y * (1.2067492 + y * (0.2659732 + y * (0.0360768 + y * 0.0045813)))));
    }
    double y = 3.75 / ax;
    return (Math.exp(ax) / Math.sqrt(ax)) * (0.39894228 + y * (0.01328592 + y * (0.00225319 + y * (-0.00157565 + y * (0.00916281 + y * (-0.02057706 + y * (0.02635537 + y * (-0.01647633 + y * 0.00392377))))))));
}

double[] window1D(String kind, int n, double p) {
    double[] w = new double[n];
    if (n <= 1) {
        Arrays.fill(w, 1.0);
        return w;
    }

    if ("tukey".equals(kind)) {
        double alpha = p;
        if (alpha <= 0) {
            Arrays.fill(w, 1.0);
            return w;
        }
        if (alpha >= 1.0) {
            for (int i = 0; i < n; i++) w[i] = 0.5 * (1.0 - Math.cos(2.0 * Math.PI * i / (n - 1.0)));
            return w;
        }
        for (int i = 0; i < n; i++) {
            double x = i / (double)(n - 1);
            if (x < alpha / 2.0) {
                w[i] = 0.5 * (1.0 + Math.cos(Math.PI * (2.0 * x / alpha - 1.0)));
            } else if (x <= 1.0 - alpha / 2.0) {
                w[i] = 1.0;
            } else {
                w[i] = 0.5 * (1.0 + Math.cos(Math.PI * (2.0 * x / alpha - 2.0 / alpha + 1.0)));
            }
        }
        return w;
    }

    if ("kaiser".equals(kind)) {
        double beta = p;
        double den = i0Bessel(beta);
        for (int i = 0; i < n; i++) {
            double r = 2.0 * i / (n - 1.0) - 1.0;
            w[i] = i0Bessel(beta * Math.sqrt(Math.max(0.0, 1.0 - r * r))) / den;
        }
        return w;
    }

    for (int i = 0; i < n; i++) {
        double t = 2.0 * Math.PI * i / (n - 1.0);
        if ("hann".equals(kind)) {
            w[i] = 0.5 - 0.5 * Math.cos(t);
        } else if ("hamming".equals(kind)) {
            w[i] = 0.54 - 0.46 * Math.cos(t);
        } else if ("blackman".equals(kind)) {
            w[i] = 0.42 - 0.5 * Math.cos(t) + 0.08 * Math.cos(2.0 * t);
        } else if ("blackmanharris".equals(kind)) {
            w[i] = 0.35875 - 0.48829 * Math.cos(t) + 0.14128 * Math.cos(2.0 * t) - 0.01168 * Math.cos(3.0 * t);
        } else if ("nuttall".equals(kind)) {
            w[i] = 0.355768 - 0.487396 * Math.cos(t) + 0.144232 * Math.cos(2.0 * t) - 0.012604 * Math.cos(3.0 * t);
        } else {
            w[i] = 1.0;
        }
    }
    return w;
}

double[][] separable2D(String kind, int h, int w, double param) {
    double[] wy = window1D(kind, h, param);
    double[] wx = window1D(kind, w, param);
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) out[y][x] = wy[y] * wx[x];
    }
    return out;
}

double[][] poissonMask2D(int h, int w, double lambdaPx) {
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        int dyEdge = Math.min(y, h - 1 - y);
        for (int x = 0; x < w; x++) {
            int dxEdge = Math.min(x, w - 1 - x);
            int d = Math.min(dxEdge, dyEdge);
            out[y][x] = 1.0 - Math.exp(-d / Math.max(1e-9, lambdaPx));
        }
    }
    return out;
}

double[][] gaussian2DRadial(int h, int w, double sigmaFrac) {
    double[][] out = new double[h][w];
    double cx = (w - 1) / 2.0;
    double cy = (h - 1) / 2.0;
    double sigma = Math.max(1e-9, sigmaFrac * Math.min(w, h));
    double twoSig2 = 2.0 * sigma * sigma;
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double dx = x - cx;
            double dy = y - cy;
            out[y][x] = Math.exp(-(dx * dx + dy * dy) / twoSig2);
        }
    }
    return out;
}

double[][] mul(double[][] a, double[][] b) {
    int h = a.length, w = a[0].length;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) out[y][x] = a[y][x] * b[y][x];
    }
    return out;
}

double[][] applyMask(double[][] img, double[][] mask) {
    int h = img.length, w = img[0].length;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) out[y][x] = img[y][x] * mask[y][x];
    }
    return out;
}

record MaskStats(double mean, double std, double p10, double p90, double edgeMean, double centerMean, double area50, double quickScore) {}

MaskStats summarizeMask(double[][] m) {
    int h = m.length, w = m[0].length;
    int n = h * w;
    double[] vals = new double[n];
    double sum = 0.0, sum2 = 0.0;
    int k = 0;

    int band = Math.max(1, (int)Math.round(0.1 * Math.min(h, w)));
    double edgeSum = 0.0, centerSum = 0.0;
    int edgeN = 0, centerN = 0, area50N = 0;

    int y0 = h / 4, y1 = h - h / 4;
    int x0 = w / 4, x1 = w - w / 4;

    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double v = m[y][x];
            vals[k++] = v;
            sum += v;
            sum2 += v * v;
            if (v >= 0.5) area50N++;

            boolean edge = (x < band || y < band || x >= w - band || y >= h - band);
            if (edge) { edgeSum += v; edgeN++; }

            boolean center = (x >= x0 && x < x1 && y >= y0 && y < y1);
            if (center) { centerSum += v; centerN++; }
        }
    }

    Arrays.sort(vals);
    double mean = sum / n;
    double var = Math.max(0.0, sum2 / n - mean * mean);
    double std = Math.sqrt(var);
    double p10 = vals[(int)Math.floor(0.10 * (n - 1))];
    double p90 = vals[(int)Math.floor(0.90 * (n - 1))];
    double edgeMean = edgeN == 0 ? Double.NaN : edgeSum / edgeN;
    double centerMean = centerN == 0 ? Double.NaN : centerSum / centerN;
    double area50 = 100.0 * area50N / n;

    double edgeSupp = 1.0 - edgeMean;
    double quickScore = 0.60 * edgeSupp + 0.40 * centerMean;

    return new MaskStats(mean, std, p10, p90, edgeMean, centerMean, area50, quickScore);
}

void drawFitLocal(Graphics2D g, BufferedImage src, int x, int y, int w, int h) {
    double sx = w / (double) src.getWidth();
    double sy = h / (double) src.getHeight();
    double s = Math.min(sx, sy);
    int nw = Math.max(1, (int)Math.round(src.getWidth() * s));
    int nh = Math.max(1, (int)Math.round(src.getHeight() * s));
    int ox = x + (w - nw) / 2;
    int oy = y + (h - nh) / 2;
    g.drawImage(src, ox, oy, nw, nh, null);
}

BufferedImage makeGridPanel(String title, java.util.List<String> names, java.util.List<BufferedImage> imgs, int columns, int cellW, int cellH) {
    int cols = Math.max(1, columns);
    int rows = (int)Math.ceil(names.size() / (double)cols);
    int gap = 14;
    int margin = 20;
    int top = 64;
    int labelH = 18;
    int w = margin * 2 + cols * cellW + (cols - 1) * gap;
    int h = top + rows * (cellH + labelH + gap) + 24;

    BufferedImage canvas = new BufferedImage(w, h, BufferedImage.TYPE_INT_RGB);
    Graphics2D g = canvas.createGraphics();
    g.setRenderingHint(RenderingHints.KEY_ANTIALIASING, RenderingHints.VALUE_ANTIALIAS_ON);
    g.setColor(Color.WHITE);
    g.fillRect(0, 0, w, h);

    g.setColor(new Color(18, 42, 76));
    g.setFont(new Font("SansSerif", Font.BOLD, 21));
    g.drawString(title, margin, 34);

    g.setFont(new Font("SansSerif", Font.PLAIN, 12));
    for (int i = 0; i < names.size(); i++) {
        int r = i / cols;
        int c = i % cols;
        int x = margin + c * (cellW + gap);
        int y = top + r * (cellH + labelH + gap);

        g.setColor(new Color(40, 40, 40));
        g.drawString(names.get(i), x + 2, y + 12);
        int iy = y + labelH;
        g.setColor(new Color(234, 234, 234));
        g.fillRect(x - 1, iy - 1, cellW + 2, cellH + 2);
        drawFitLocal(g, imgs.get(i), x, iy, cellW, cellH);
    }

    g.dispose();
    return canvas;
}

Path rootMaskCmp = findProjectRootMaskCmp(Paths.get(System.getProperty("user.dir")));
if (rootMaskCmp == null) throw new RuntimeException("Could not locate project root for FFT mask comparison.");

Path outDir = rootMaskCmp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("filter_mask_comparison");
Files.createDirectories(outDir);

java.util.List<String> bases = Arrays.asList("C15D5P001_1", "Picture1");
java.util.List<Path> caseDirs = Arrays.asList(
    rootMaskCmp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootMaskCmp.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

// Tunables
final double POISSON_LAMBDA_PX = 18.0;
final double TUKEY_ALPHA = 0.35;
final double KAISER_BETA = 8.0;
final double GAUSS_SIGMA_FRAC = 0.24;

Map<String, java.util.List<Double>> allQuickScores = new LinkedHashMap<>();

for (int ci = 0; ci < bases.size(); ci++) {
    String base = bases.get(ci);
    Path caseDir = caseDirs.get(ci);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    java.util.List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(caseDir, "*_crop.jpg")) {
        for (Path f : ds) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    if (ids.isEmpty()) throw new RuntimeException("No crop tiles found for " + base + " in " + caseDir + ". Run generation cells first.");

    int tile = ids.contains(4) ? 4 : ids.get(0);
    Path tilePath = caseDir.resolve(base + "_tile" + tile + "_crop.jpg");
    BufferedImage tileImg = ImageIO.read(tilePath.toFile());
    if (tileImg == null) throw new RuntimeException("Could not read tile image: " + tilePath);

    double[][] img = toGray01(tileImg);
    int h = img.length, w = img[0].length;

    double[][] mPoisson = poissonMask2D(h, w, POISSON_LAMBDA_PX);
    double[][] mHann = separable2D("hann", h, w, 0.0);
    double[][] mHamming = separable2D("hamming", h, w, 0.0);
    double[][] mBlackman = separable2D("blackman", h, w, 0.0);
    double[][] mBlackmanHarris = separable2D("blackmanharris", h, w, 0.0);
    double[][] mNuttall = separable2D("nuttall", h, w, 0.0);
    double[][] mKaiser = separable2D("kaiser", h, w, KAISER_BETA);
    double[][] mTukey = separable2D("tukey", h, w, TUKEY_ALPHA);
    double[][] mBoth = mul(mPoisson, mTukey);
    double[][] mGauss = gaussian2DRadial(h, w, GAUSS_SIGMA_FRAC);

    LinkedHashMap<String, double[][]> masks = new LinkedHashMap<>();
    masks.put("Poisson", mPoisson);
    masks.put("Hann2D", mHann);
    masks.put("Hamming2D", mHamming);
    masks.put("Blackman2D", mBlackman);
    masks.put("BlackmanHarris2D", mBlackmanHarris);
    masks.put("Nuttall2D", mNuttall);
    masks.put("Kaiser2D", mKaiser);
    masks.put("Tukey2D", mTukey);
    masks.put("Both(P*T)", mBoth);
    masks.put("Gaussian2D", mGauss);

    java.util.List<String> names = new ArrayList<>(masks.keySet());
    java.util.List<BufferedImage> maskImgs = new ArrayList<>();
    java.util.List<BufferedImage> maskedImgs = new ArrayList<>();
    Map<String, MaskStats> stats = new LinkedHashMap<>();

    for (String k : names) {
        double[][] mm = masks.get(k);
        maskImgs.add(colorizeMask(mm));
        maskedImgs.add(gray01ToImage(applyMask(img, mm)));
        MaskStats s = summarizeMask(mm);
        stats.put(k, s);
        allQuickScores.computeIfAbsent(k, kk -> new ArrayList<>()).add(s.quickScore());
    }

    BufferedImage panelMasks = makeGridPanel(base + " - Mask shapes", names, maskImgs, 4, 210, 155);
    BufferedImage panelMasked = makeGridPanel(base + " - Masked tile preview (tile " + tile + ")", names, maskedImgs, 4, 210, 155);

    Path outMasks = outDir.resolve(base + "_mask_shapes_comparison.png");
    Path outMasked = outDir.resolve(base + "_masked_tile_comparison.png");
    ImageIO.write(panelMasks, "png", outMasks.toFile());
    ImageIO.write(panelMasked, "png", outMasked.toFile());

    System.out.println("\n=================================================");
    System.out.println("Dataset: " + base);
    System.out.println("Input tile: " + tilePath);
    System.out.println("Masks panel: " + outMasks);
    System.out.println("Masked panel: " + outMasked);
    System.out.println("name\tquickScore\tmean\tstd\tedgeMean\tcenterMean\tarea>=0.5(%)");

    java.util.List<Map.Entry<String, MaskStats>> sorted = new ArrayList<>(stats.entrySet());
    sorted.sort((a, b) -> Double.compare(b.getValue().quickScore(), a.getValue().quickScore()));
    for (Map.Entry<String, MaskStats> e : sorted) {
        MaskStats s = e.getValue();
        System.out.printf(Locale.US, "%s\t%.4f\t%.4f\t%.4f\t%.4f\t%.4f\t%.2f%n",
            e.getKey(), s.quickScore(), s.mean(), s.std(), s.edgeMean(), s.centerMean(), s.area50());
    }

    System.out.println("Top 3 (quick): " + sorted.get(0).getKey() + ", " + sorted.get(1).getKey() + ", " + sorted.get(2).getKey());
}

System.out.println("\n=================================================");
System.out.println("Cross-dataset mean quick score (higher is better)");
java.util.List<Map.Entry<String, java.util.List<Double>>> avgList = new ArrayList<>(allQuickScores.entrySet());
avgList.sort((a, b) -> {
    double ma = a.getValue().stream().mapToDouble(v -> v).average().orElse(Double.NaN);
    double mb = b.getValue().stream().mapToDouble(v -> v).average().orElse(Double.NaN);
    return Double.compare(mb, ma);
});
for (Map.Entry<String, java.util.List<Double>> e : avgList) {
    double meanQ = e.getValue().stream().mapToDouble(v -> v).average().orElse(Double.NaN);
    System.out.printf(Locale.US, "%s\tmeanQuick=%.4f\tn=%d%n", e.getKey(), meanQ, e.getValue().size());
}

System.out.println("\nQuick score definition:");
System.out.println("quickScore = 0.60*(1-edgeMean) + 0.40*(centerMean)");
System.out.println("Parameters: lambda=" + POISSON_LAMBDA_PX + ", tukey.alpha=" + TUKEY_ALPHA + ", kaiser.beta=" + KAISER_BETA + ", gauss.sigmaFrac=" + GAUSS_SIGMA_FRAC);
System.out.println("Done ✅");


Dataset: C15D5P001_1
Input tile: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\generated_from_source\C15D5P001_1_tile4_crop.jpg
Masks panel: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\filter_mask_comparison\C15D5P001_1_mask_shapes_comparison.png
Masked panel: c:\Users\dunnmk\repos\imgjplugin\notebooks\_assets\fiba_tile_montage\filter_mask_comparison\C15D5P001_1_masked_tile_comparison.png
name	quickScore	mean	std	edgeMean	centerMean	area>=0.5(%)
Both(P*T)	0.9137	0.4936	0.3525	0.0744	0.8959	53.86
Tukey2D	0.8863	0.6700	0.3898	0.1895	1.0000	65.60
Hann2D	0.8563	0.2461	0.2790	0.0165	0.6656	19.90
Hamming2D	0.8382	0.2877	0.2697	0.0629	0.6898	21.97
Kaiser2D	0.8195	0.1869	0.2521	0.0093	0.5628	13.89
Gaussian2D	0.8195	0.3355	0.2593	0.1104	0.7143	25.10
Blackman2D	0.8129	0.1737	0.2473	0.0055	0.5405	12.89
Poisson	0.8035	0.5798	0.2807	0.2581	0.8959	63.50
BlackmanHarris2D	0.7755	0.1267	0.2224	0.0011	0.4404	9.23
Nuttall2D	0.7736	0.1246	0.2211	0.0009	0.43

### Generated window/mask comparison panels

C15D5P001_1 mask shapes:

![](_assets/fiba_tile_montage/filter_mask_comparison/C15D5P001_1_mask_shapes_comparison.png)

C15D5P001_1 masked tile effect:

![](_assets/fiba_tile_montage/filter_mask_comparison/C15D5P001_1_masked_tile_comparison.png)

Picture1 mask shapes:

![](_assets/fiba_tile_montage/filter_mask_comparison/Picture1_mask_shapes_comparison.png)

Picture1 masked tile effect:

![](_assets/fiba_tile_montage/filter_mask_comparison/Picture1_masked_tile_comparison.png)

Use the **Quick score** printed by the code cell as the fast ranking signal across window families.

In [17]:
import java.awt.image.BufferedImage;
import javax.imageio.ImageIO;
import java.nio.file.*;
import java.util.*;
import java.util.regex.*;

record MaskRecScore(double maskSeparation, double corrWithRec, double totalScore, double edgeMean, double centerMean, double corrRaw) {}

double[][] gray01Obj(BufferedImage img) {
    int h = img.getHeight(), w = img.getWidth();
    double[][] g = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            int rgb = img.getRGB(x, y);
            int r = (rgb >> 16) & 0xff;
            int gg = (rgb >> 8) & 0xff;
            int b = rgb & 0xff;
            g[y][x] = (0.299 * r + 0.587 * gg + 0.114 * b) / 255.0;
        }
    }
    return g;
}

double[][] applyObjMask(double[][] img, double[][] mask) {
    int h = img.length, w = img[0].length;
    double[][] out = new double[h][w];
    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) out[y][x] = img[y][x] * mask[y][x];
    }
    return out;
}

double[] maskEdgeCenterMeans(double[][] mask) {
    int h = mask.length, w = mask[0].length;
    int band = Math.max(2, (int)Math.round(0.1 * Math.min(h, w)));
    int y0 = h / 4, y1 = h - h / 4;
    int x0 = w / 4, x1 = w - w / 4;

    double edgeSum = 0.0, centerSum = 0.0;
    int edgeN = 0, centerN = 0;

    for (int y = 0; y < h; y++) {
        for (int x = 0; x < w; x++) {
            double v = mask[y][x];
            boolean edge = (x < band || y < band || x >= w - band || y >= h - band);
            if (edge) {
                edgeSum += v;
                edgeN++;
            }
            boolean center = (x >= x0 && x < x1 && y >= y0 && y < y1);
            if (center) {
                centerSum += v;
                centerN++;
            }
        }
    }

    double edgeMean = edgeSum / Math.max(1, edgeN);
    double centerMean = centerSum / Math.max(1, centerN);
    return new double[] {edgeMean, centerMean};
}

double pearsonCorrCenter(double[][] a, double[][] b) {
    int h = a.length, w = a[0].length;
    int y0 = h / 4, y1 = h - h / 4;
    int x0 = w / 4, x1 = w - w / 4;

    double sa = 0.0, sb = 0.0;
    int n = 0;
    for (int y = y0; y < y1; y++) {
        for (int x = x0; x < x1; x++) {
            sa += a[y][x];
            sb += b[y][x];
            n++;
        }
    }
    if (n < 2) return 0.0;
    double ma = sa / n;
    double mb = sb / n;

    double num = 0.0, da2 = 0.0, db2 = 0.0;
    for (int y = y0; y < y1; y++) {
        for (int x = x0; x < x1; x++) {
            double da = a[y][x] - ma;
            double db = b[y][x] - mb;
            num += da * db;
            da2 += da * da;
            db2 += db * db;
        }
    }

    double den = Math.sqrt(Math.max(1e-12, da2) * Math.max(1e-12, db2));
    return num / den;
}

MaskRecScore calcMaskRecScore(double[][] ref, double[][] mask) {
    double[][] rec = applyObjMask(ref, mask);

    double[] edgeCenter = maskEdgeCenterMeans(mask);
    double edgeMean = edgeCenter[0];
    double centerMean = edgeCenter[1];

    double maskSeparation = Math.max(0.0, Math.min(1.0, centerMean - edgeMean));

    double corrRaw = pearsonCorrCenter(ref, rec);
    double corrWithRec = Math.max(0.0, Math.min(1.0, 0.5 * (corrRaw + 1.0)));

    double total = 0.70 * maskSeparation + 0.30 * corrWithRec;
    return new MaskRecScore(maskSeparation, corrWithRec, total, edgeMean, centerMean, corrRaw);
}

Path rootObj = findProjectRootMaskCmp(Paths.get(System.getProperty("user.dir")));
if (rootObj == null) throw new RuntimeException("Could not find project root for unified objective scoring.");

java.util.List<String> basesObj = Arrays.asList("C15D5P001_1", "Picture1");
java.util.List<Path> dirsObj = Arrays.asList(
    rootObj.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source"),
    rootObj.resolve("notebooks").resolve("_assets").resolve("fiba_tile_montage").resolve("generated_from_source_picture1")
);

final double POISSON_LAMBDA_PX_OBJ = 18.0;
final double TUKEY_ALPHA_OBJ = 0.35;
final double KAISER_BETA_OBJ = 8.0;
final double GAUSS_SIGMA_FRAC_OBJ = 0.24;

Map<String, java.util.List<Double>> meanUnified = new LinkedHashMap<>();

for (int ci = 0; ci < basesObj.size(); ci++) {
    String base = basesObj.get(ci);
    Path dir = dirsObj.get(ci);

    Pattern p = Pattern.compile("^" + Pattern.quote(base) + "_tile(\\d+)_crop\\.jpg$");
    java.util.List<Integer> ids = new ArrayList<>();
    try (DirectoryStream<Path> ds = Files.newDirectoryStream(dir, "*_crop.jpg")) {
        for (Path f : ds) {
            Matcher m = p.matcher(f.getFileName().toString());
            if (m.matches()) ids.add(Integer.parseInt(m.group(1)));
        }
    }
    ids.sort(Comparator.naturalOrder());
    if (ids.isEmpty()) throw new RuntimeException("No crop tiles found for " + base + ".");

    int tile = ids.contains(4) ? 4 : ids.get(0);
    Path tilePath = dir.resolve(base + "_tile" + tile + "_crop.jpg");
    BufferedImage imgB = ImageIO.read(tilePath.toFile());
    if (imgB == null) throw new RuntimeException("Cannot read tile: " + tilePath);

    double[][] ref = gray01Obj(imgB);
    int h = ref.length, w = ref[0].length;

    LinkedHashMap<String, double[][]> masks = new LinkedHashMap<>();
    double[][] mPoisson = poissonMask2D(h, w, POISSON_LAMBDA_PX_OBJ);
    double[][] mTukey = separable2D("tukey", h, w, TUKEY_ALPHA_OBJ);
    masks.put("Poisson", mPoisson);
    masks.put("Hann2D", separable2D("hann", h, w, 0.0));
    masks.put("Hamming2D", separable2D("hamming", h, w, 0.0));
    masks.put("Blackman2D", separable2D("blackman", h, w, 0.0));
    masks.put("BlackmanHarris2D", separable2D("blackmanharris", h, w, 0.0));
    masks.put("Nuttall2D", separable2D("nuttall", h, w, 0.0));
    masks.put("Kaiser2D", separable2D("kaiser", h, w, KAISER_BETA_OBJ));
    masks.put("Tukey2D", mTukey);
    masks.put("Both(P*T)", mul(mPoisson, mTukey));
    masks.put("Gaussian2D", gaussian2DRadial(h, w, GAUSS_SIGMA_FRAC_OBJ));

    Map<String, MaskRecScore> scores = new LinkedHashMap<>();
    for (Map.Entry<String, double[][]> e : masks.entrySet()) {
        MaskRecScore us = calcMaskRecScore(ref, e.getValue());
        scores.put(e.getKey(), us);
        meanUnified.computeIfAbsent(e.getKey(), k -> new ArrayList<>()).add(us.totalScore());
    }

    java.util.List<Map.Entry<String, MaskRecScore>> ranked = new ArrayList<>(scores.entrySet());
    ranked.sort((a, b) -> Double.compare(b.getValue().totalScore(), a.getValue().totalScore()));

    System.out.println("\n=================================================");
    System.out.println("Unified objective metric | Dataset=" + base + " | tile=" + tile);
    System.out.println("method\tU_total\tmaskSeparation\tcorrWithRec\tedgeMean\tcenterMean\tcorrRaw");
    for (Map.Entry<String, MaskRecScore> e : ranked) {
        MaskRecScore s = e.getValue();
        System.out.printf(Locale.US, "%s\t%.4f\t%.4f\t%.4f\t%.4f\t%.4f\t%.4f%n",
            e.getKey(), s.totalScore(), s.maskSeparation(), s.corrWithRec(), s.edgeMean(), s.centerMean(), s.corrRaw());
    }
    System.out.println("Top 3 unified: " + ranked.get(0).getKey() + ", " + ranked.get(1).getKey() + ", " + ranked.get(2).getKey());
}

System.out.println("\n=================================================");
System.out.println("Cross-dataset mean unified score");
java.util.List<Map.Entry<String, java.util.List<Double>>> all = new ArrayList<>(meanUnified.entrySet());
all.sort((a, b) -> Double.compare(
    b.getValue().stream().mapToDouble(v -> v).average().orElse(Double.NaN),
    a.getValue().stream().mapToDouble(v -> v).average().orElse(Double.NaN)
));
for (Map.Entry<String, java.util.List<Double>> e : all) {
    double m = e.getValue().stream().mapToDouble(v -> v).average().orElse(Double.NaN);
    System.out.printf(Locale.US, "%s\tmeanUnified=%.4f\tn=%d%n", e.getKey(), m, e.getValue().size());
}

System.out.println("\nScore definition:");
System.out.println("score = 0.70*(mask-separation) + 0.30*(corr-with-rec)");
System.out.println("mask-separation = clip(centerMean(mask) - edgeMean(mask), 0, 1)");
System.out.println("corr-with-rec = clip((corrRaw + 1)/2, 0, 1), where corrRaw is Pearson corr(ref, rec) in center ROI");
System.out.println("Done unified objective scoring ✅");


Unified objective metric | Dataset=C15D5P001_1 | tile=4
method	U_total	maskSeparation	corrWithRec	edgeMean	centerMean	corrRaw
Both(P*T)	0.8748	0.8215	0.9992	0.0744	0.8959	0.9985
Tukey2D	0.8673	0.8105	1.0000	0.1895	1.0000	1.0000
Hann2D	0.7463	0.6491	0.9732	0.0165	0.6656	0.9463
Poisson	0.7463	0.6379	0.9992	0.2581	0.8959	0.9985
Hamming2D	0.7321	0.6269	0.9776	0.0629	0.6898	0.9551
Gaussian2D	0.7173	0.6040	0.9818	0.1104	0.7143	0.9637
Kaiser2D	0.6730	0.5535	0.9519	0.0093	0.5628	0.9037
Blackman2D	0.6584	0.5350	0.9461	0.0055	0.5405	0.8923
BlackmanHarris2D	0.5823	0.4393	0.9160	0.0011	0.4404	0.8320
Nuttall2D	0.5784	0.4345	0.9143	0.0009	0.4354	0.8286
Top 3 unified: Both(P*T), Tukey2D, Hann2D

Unified objective metric | Dataset=Picture1 | tile=4
method	U_total	maskSeparation	corrWithRec	edgeMean	centerMean	corrRaw
Both(P*T)	0.8748	0.8215	0.9992	0.0744	0.8959	0.9984
Tukey2D	0.8673	0.8105	1.0000	0.1895	1.0000	1.0000
Poisson	0.7463	0.6379	0.9992	0.2581	0.8959	0.9984
Hann2D	0.7454	0.6491	0.9701	0.0165

## Unified objective metric (user-corrected)

Using your corrected target directly:

$$
\text{score}=0.7\cdot(\text{mask-separation})+0.3\cdot(\text{corr-with-rec})
$$

Definitions used in the scoring cell:

- $\text{mask-separation}=\mathrm{clip}(\mu_{\text{center}}(M)-\mu_{\text{edge}}(M),0,1)$
- $\text{corr-with-rec}=\mathrm{clip}\left(\frac{r+1}{2},0,1\right)$ where $r$ is Pearson correlation between reference image and reconstructed image in the center ROI.

This keeps each term on $[0,1]$ so the weighted sum is stable and interpretable.